# Encoding Technique 4: Frequency Encoding

**Dataset:** `Loan_Default.csv`

**When to use:** For **nominal** features, especially when dealing with **high-cardinality** columns where OHE would create too many columns.

**Key concept:** Each category is replaced by the **proportion** (frequency) with which it appears in the dataset. If `'Male'` appears in 30% of rows, every `'Male'` entry is replaced by `0.30`.

---

### Step 1: Setup, Data Loading & Prep

In [ ]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

# Load data
df = pd.read_csv('../../data/raw/Loan_Default.csv')
df.drop(['ID', 'year'], axis=1, inplace=True)

categorical_features = df.select_dtypes(include=['object']).columns.tolist()
Ordinal_features = ['age']
Nominal_features = categorical_features.copy()
Nominal_features.remove('age')

# Encode the ordinal feature first
enc = OrdinalEncoder()
df[Ordinal_features] = enc.fit_transform(df[Ordinal_features])

print(f'Shape before Frequency Encoding: {df.shape}')
df.head()

### Step 2: Apply Frequency Encoding

For each nominal column, we replace the label with its proportion in the dataset. This keeps the dimensionality constant.

In [ ]:
df_freq = df.copy()

for c in Nominal_features:
    # Replace category with its frequency (proportion)
    df_freq[c + '_freq'] = df_freq[c].map(df_freq.groupby(c).size() / df_freq.shape[0])

    # Replace original column with integer codes (intermediate step)
    indexer = pd.factorize(df_freq[c], sort=True)[1]
    df_freq[c] = indexer.get_indexer(df_freq[c])

# Drop original columns, keep only the _freq versions
df_freq = df_freq.drop(Nominal_features, axis=1)

print(f'Shape after Frequency Encoding: {df_freq.shape}')
df_freq.head()

### Step 3: Inspect the Frequency Values

Check the frequency values for a feature like `Gender`.

In [ ]:
print(df_freq['Gender_freq'].value_counts())

### Key Observation

The dimensionality remains the same. Frequency encoding is highly efficient but can lead to collisions if two different categories have identical frequencies.